# Day 6 Mid-session Exercise - build the tool-use loop (auto-graded)

Offline by default - no AWS needed. Fill the **6 TODOs**, then run the **scoreboard** cell at the bottom; it asserts each step and prints your score out of 6.

Anchor: TravelMind UC-A1. One tool: `lookup_booking`.

**How to work it:** complete the TODOs top to bottom, then run the last cell. Set `USE_REAL_BEDROCK = True` only if you want to run the same code against real Bedrock.

In [ ]:
# ---- offline by default; flip to real Bedrock when you want ----
USE_REAL_BEDROCK = False
REGION = "us-west-2"
MODEL  = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"
import json

In [ ]:
# ---- provided scaffolding: mock, tool, client. Do not edit. ----
EXEC_COUNT = 0
def lookup_booking(pnr):
    global EXEC_COUNT; EXEC_COUNT += 1
    return {"pnr": pnr, "status": "CANCELLED", "flight": "AI-302", "date": "2026-06-12"}
TOOLS = {"lookup_booking": lookup_booking}
SYSTEM = "You are TravelMind. Use the tool to look up a booking before answering. Never invent a PNR."

class MockBedrock:
    """Offline stand-in for bedrock-runtime.converse: asks for the tool once, then answers."""
    def converse(self, modelId=None, messages=None, toolConfig=None, system=None, inferenceConfig=None, **kw):
        n = sum(1 for m in messages for c in m.get("content", []) if isinstance(c, dict) and "toolResult" in c)
        if toolConfig and n == 0:
            return {"output": {"message": {"role": "assistant", "content": [
                        {"toolUse": {"toolUseId": "tu_1", "name": "lookup_booking", "input": {"pnr": "JX48Q2"}}}]}},
                    "stopReason": "tool_use", "usage": {}}
        return {"output": {"message": {"role": "assistant", "content": [
                    {"text": "Booking JX48Q2: flight AI-302 is CANCELLED due to weather. "
                             "Rebooking options: AI-318 at 18:40 or 6E-552 at 21:15."}]}},
                "stopReason": "end_turn", "usage": {}}

class RunawayMock:
    """Never stops asking for a tool - used only to test your guard. Raises if your loop never stops."""
    def __init__(self): self.calls = 0
    def converse(self, **kw):
        self.calls += 1
        if self.calls > 30:
            raise RuntimeError("runaway loop: your guard is missing")
        return {"output": {"message": {"role": "assistant", "content": [
                    {"toolUse": {"toolUseId": "tu_x", "name": "lookup_booking", "input": {"pnr": "JX48Q2"}}}]}},
                "stopReason": "tool_use", "usage": {}}

_OVERRIDE = None
def get_client():
    if _OVERRIDE is not None:
        return _OVERRIDE
    if USE_REAL_BEDROCK:
        import boto3
        return boto3.client("bedrock-runtime", region_name=REGION)
    return MockBedrock()

print("ready -", "REAL Bedrock" if USE_REAL_BEDROCK else "offline mock")

### TODO 1 - define a `toolSpec`

Build `tool_config` with ONE tool. A tool spec = `name` + `description` + an `inputSchema` (JSON schema) declaring a required string `pnr`.

In [ ]:
# TODO 1: define tool_config with one toolSpec for lookup_booking.
#   name "lookup_booking", a useful description, and inputSchema.json = an object
#   with a string property "pnr" that is required.
tool_config = {"tools": []}   # <-- replace this

### TODO 2 - send the first request

Call `converse` with the user message and your `tool_config`. The model should reply with `stopReason == "tool_use"`.

In [ ]:
messages = [{"role": "user", "content": [{"text": "What is the status of PNR JX48Q2?"}]}]
# TODO 2: call converse and store the response in `resp`.
#   get_client().converse(modelId=MODEL, system=[{"text": SYSTEM}], messages=messages,
#                         toolConfig=tool_config, inferenceConfig={"maxTokens": 512})
resp = None   # <-- replace this
print("stopReason:", None if resp is None else resp["stopReason"])

### TODO 3 - append the assistant message

Before you can return a tool result, the assistant's `tool_use` message must be in `messages`. This is the step everyone forgets.

In [ ]:
# TODO 3: append the assistant tool_use message (resp's message) to `messages`.
# ... your one line here ...
print("messages so far:", len(messages))

### TODO 4 - build the `toolResult` with a matching id

Run the tool the model asked for, then return its output in a **user** turn whose `toolUseId` matches the `toolUse` id.

In [ ]:
# TODO 4: using the tool the model asked for:
#   tu = resp["output"]["message"]["content"][-1]["toolUse"]
#   out = TOOLS[tu["name"]](**tu["input"])
#   then append a USER turn with a toolResult: toolUseId = tu["toolUseId"], content = [{"json": out}]
# ... your code here ...
print("last turn role:", messages[-1]["role"])

### TODO 5 and 6 - the loop, and the guard

Generalise the one-turn work into a loop. **TODO 5:** loop until `stopReason` is no longer `tool_use`. **TODO 6:** add a guard so that if the loop reaches `max_steps`, it returns a short "step limit" message instead of looping forever.

In [ ]:
def run_agent(user_text, max_steps=6):
    messages = [{"role": "user", "content": [{"text": user_text}]}]
    resp = get_client().converse(modelId=MODEL, system=[{"text": SYSTEM}], messages=messages,
                                 toolConfig=tool_config, inferenceConfig={"maxTokens": 512})
    steps = 0
    # TODO 5: loop while resp["stopReason"] == "tool_use":
    #   - TODO 6 (guard): if steps >= max_steps, return a short "step limit" message
    #   - steps += 1
    #   - tu = resp["output"]["message"]["content"][-1]["toolUse"]
    #   - messages.append(resp["output"]["message"])          # assistant turn first
    #   - out = TOOLS[tu["name"]](**tu["input"])
    #   - append the toolResult (user turn, matching id)
    #   - resp = get_client().converse(... messages ...)        # go again
    return resp["output"]["message"]["content"][0]["text"]

# try it once TODO 5/6 are done:
# print(run_agent("What is the status of PNR JX48Q2?"))

### Scoreboard

Run this last. It asserts each step and prints your score out of 6.

In [ ]:
# ---- assert-based scoreboard. Run after completing the TODOs. ----
score = 0; total = 0
def check(label, test):
    global score, total
    total += 1
    try:
        test(); score += 1; print("  PASS  " + label)
    except Exception as e:
        print("  FAIL  " + label + "  ->  " + (str(e) or type(e).__name__))

def t1():
    ts = tool_config["tools"][0]["toolSpec"]
    assert ts["name"] == "lookup_booking", "tool name must be lookup_booking"
    js = ts["inputSchema"]["json"]
    assert "pnr" in js["properties"], "schema needs a 'pnr' property"
    assert js["required"] == ["pnr"], "pnr must be required"

def t2():
    assert resp is not None, "resp is None - call converse"
    assert resp["stopReason"] == "tool_use", "first response should be tool_use"
    assert resp["output"]["message"]["content"][-1]["toolUse"]["name"] == "lookup_booking"

def t3():
    assert any(m.get("role") == "assistant" and any("toolUse" in c for c in m["content"]) for m in messages),         "append the assistant tool_use message before the toolResult"

def t4():
    tids = [c["toolUse"]["toolUseId"] for m in messages if m.get("role") == "assistant"
            for c in m["content"] if "toolUse" in c]
    rids = [c["toolResult"]["toolUseId"] for m in messages if m.get("role") == "user"
            for c in m["content"] if "toolResult" in c]
    assert tids and rids, "need both a toolUse and a toolResult turn"
    assert tids[0] in rids, "the toolResult id must match the toolUse id"

def t5():
    global _OVERRIDE, EXEC_COUNT
    _OVERRIDE = None; EXEC_COUNT = 0
    ans = run_agent("What is the status of PNR JX48Q2?", max_steps=6)
    assert isinstance(ans, str) and ("CANCELLED" in ans or "AI-302" in ans), "the loop should return the final answer"
    assert EXEC_COUNT >= 1, "the loop should run the tool at least once"

def t6():
    global _OVERRIDE, EXEC_COUNT
    _OVERRIDE = RunawayMock(); EXEC_COUNT = 0
    try:
        run_agent("force a long run", max_steps=3)
    finally:
        _OVERRIDE = None
    assert 1 <= EXEC_COUNT <= 3, "guard missing: the loop ran away or never ran"

for lbl, fn in [("TODO 1  toolSpec for lookup_booking", t1),
                ("TODO 2  first converse returns tool_use", t2),
                ("TODO 3  assistant tool_use message appended", t3),
                ("TODO 4  toolResult built with matching id", t4),
                ("TODO 5  loop runs to a final answer", t5),
                ("TODO 6  loop has a working max-steps guard", t6)]:
    check(lbl, fn)

print("\nSCORE: " + str(score) + " / " + str(total))
print("All checks passed - nice work." if score == total else "Fix the FAILs above, then re-run this cell.")